# Step 4 · Modeling: what explains outage duration in a hurricane the model has not seen

**The question**: given a hurricane's wind and rain, and each county's land, people, flooding and satellite embeddings, can we tell **which counties stay without power longer**? How much does each kind of information contribute?

**Approach**

- **Sample**: the exposed county × event rows flagged in Step 3a that have EAGLE-I records
- **Outcome**: `hours_above` (hours with at least 10% of customers out), modeled as log(1 + hours); a separate classifier predicts whether it exceeds 24 hours
- **Validation: leave one hurricane out**. Each time 8 hurricanes are used for training and the remaining one is predicted, for all 9 in turn. Every metric is computed only on the hurricane left out of training
- **Features are added group by group**: wind → plus rain → plus place (land, terrain, people and housing) → plus flood (radar flooded share and low-lying land) → plus satellite embeddings. One more set, "wind + rain + satellite embeddings", tests whether the embeddings can replace the hand-built place variables
- **Models**: gradient-boosted trees (main model) and ridge regression (linear baseline). The settings are fixed in cell 2 beforehand and are not tuned on the results

**Main metric**: within the held-out hurricane, the Spearman rank correlation between predicted and actual outage duration, averaged over the 9 hurricanes. It answers "can the model rank the counties of a new hurricane by how long they lose power".

**This is an explanatory model, not a forecast**: wind and rain are observed values after the fact, not forecast values.

**Variables kept out of the model**

- Night lights: what they see is the outage itself
- Customer totals: in Step 2c they were raised wherever the outage peak exceeded the estimate, so they carry information about the outcome
- Latitude and longitude: with them a model can memorize "which counties lost power before". How much such place memory matters is tested separately by the strict evaluation of part G

**Structure**

| Part | Content |
| --- | --- |
| A | Read the data; define the feature groups |
| B | Validation method and two baselines that need no model |
| C | Add the feature groups one by one; leave-one-hurricane-out validation |
| D | Does each group bring an improvement (decision rule fixed beforehand) |
| E | Predicted against actual |
| F | Which features matter most; gusts and forest |
| G | Robustness: other outcomes, strict evaluation, county size |
| H | An independent check with night lights |
| I | Checklist |

No Earth Engine is needed; the whole notebook takes about 3–5 minutes. The `us_flood_outage` folder must hold `step3a_features_county_event.csv`, `step3a_embeddings_county_year.csv` and `step3b_features_county_event.csv`.

In [ ]:
# 1. Environment, output folder, chart style
import os, sys, itertools
import numpy as np
import pandas as pd
import scipy, sklearn, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/us_flood_outage"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
os.makedirs(OUT, exist_ok=True)
print("Output folder:", OUT)
# record library versions: tree-model numbers differ in the second or third decimal between scikit-learn versions
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

def tidy(df, n=2):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

C1, C2 = "#2a78d6", "#eb6834"
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

The model settings, the decision rule and the thresholds of the checklist are all here, written down before any result was seen.

In [ ]:
# 2. Parameters
LONG_H, VERY_LONG_H = 24, 72     # the two thresholds (hours) for a "long outage"

# gradient-boosted trees: shallow and slow-learning, since there are only about a thousand rows
GBT = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=8, min_samples_leaf=20, l2_regularization=1.0,
           early_stopping=False, random_state=0)
RIDGE_ALPHA = 10.0               # ridge penalty (features are standardized first)
EMB_PCS     = 16                 # satellite embeddings are first reduced to this many principal components (fitted on training data only); 0 = use the 64 dimensions as they are
EMB_YEAR    = {2017: 2017, 2018: 2017, 2020: 2019, 2021: 2020, 2022: 2021, 2024: 2023}    # event year → embedding year (as in Step 3a)

# decision rule: adding a feature group "brings an improvement" only if the mean Spearman rises by at least MIN_GAIN and at least MIN_EVENTS of the 9 hurricanes improve
MIN_GAIN, MIN_EVENTS = 0.02, 6
REFERENCE = "wind_rain_place"    # reference model: used for the figures of parts E, F and H
N_PERM, N_BOOT, SEED = 20, 500, 0   # permutation repeats, bootstrap repeats, random seed

## A. Read the data; define the feature groups

The Step 3a feature table, the Step 3b flood features and the satellite embeddings are joined on `event` and `fips`; only rows that are exposed (`exposed`) and have EAGLE-I records are kept.

| Group | Content |
| --- | --- |
| `wind` | Gusts and wind speed (RTMA), distance to the track, wind speed along the track, strongest wind radius reached, side of the track |
| `rain` | Total rainfall, largest 1-day and 3-day rainfall (PRISM) |
| `land` land and terrain | Shares of forest, built-up land, wetland, cropland and impervious surface, share of low-lying land, elevation |
| `people` people and housing | Population, density, income, poverty, older residents, renters, mobile homes, no vehicle, electric heating, age of housing, vacancy, users of electricity-dependent medical equipment |
| `flood_static` low-lying land | Shares of land and of built-up land less than 2 m and 5 m above the nearest stream |
| `flood_radar` radar flood | Flooded share (three versions), day of the first image, timeliness, radar coverage |
| `embeddings` satellite embeddings | County means of the 64 AlphaEarth dimensions, reduced to 16 principal components |

"Place" means the `land` and `people` groups together. The radar flood features are empty for counties without coverage; gradient-boosted trees handle empty values directly, and ridge regression fills them with the median.

In [ ]:
# 3. Read the data, join the tables, define the feature groups and feature sets
EVENT_NAMES = {"irma2017": "Irma 2017", "florence2018": "Florence 2018", "michael2018": "Michael 2018", "laura2020": "Laura 2020",
               "zeta2020": "Zeta 2020", "ida2021": "Ida 2021", "ian2022": "Ian 2022", "beryl2024": "Beryl 2024", "helene2024": "Helene 2024"}
E_COLS = [f"e{i:02d}" for i in range(64)]

feat3a = pd.read_csv(f"{OUT}/step3a_features_county_event.csv", dtype={"fips": str})
feat3b = pd.read_csv(f"{OUT}/step3b_features_county_event.csv", dtype={"fips": str}).drop(columns="name")
emb    = pd.read_csv(f"{OUT}/step3a_embeddings_county_year.csv", dtype={"fips": str})

df = feat3a[feat3a["exposed"] & ~feat3a["no_record"] & feat3a["event"].isin(list(EVENT_NAMES))].merge(feat3b, on=["event", "fips"], how="left")
df["emb_year"] = df["year"].map(EMB_YEAR)
df = df.merge(emb[["fips", "year"] + E_COLS].rename(columns={"year": "emb_year"}), on=["fips", "emb_year"], how="left").reset_index(drop=True).copy()
EVENTS = [k for k in EVENT_NAMES if (df["event"] == k).any()]

df["y"]      = np.log1p(df["hours_above"])                          # main outcome: log(1 + hours)
df["y_alt"]  = np.log1p(df["hours_above_alt"])                      # second denominator
df["y_rel"]  = np.log1p(df["hours_rel"])                            # relative duration, affected counties only
df["long"]   = (df["hours_above"] >= LONG_H).astype(int)
df["log_pop"]         = np.log10(df["pop"])
df["log_pop_density"] = np.log10(df["pop_density"])
df["right_of_track"]  = df["right_of_track"].astype(int)
df["s1_timely"]       = df["s1_timely"].astype(float)

GROUPS = {
    "wind":   ["gust_mean_ms", "gust_peak_ms", "wind_mean_ms", "wind_swath_kt", "track_dist_km", "track_vmax_kt", "closest_hours", "right_of_track"],
    "rain":   ["rain_total_mm", "rain_max1d_mm", "rain_max3d_mm"],
    "land":   ["forest_pct", "developed_pct", "wetland_pct", "farmland_pct", "impervious_pct", "low_lying_pct", "elevation_m"],
    "people": ["log_pop", "log_pop_density", "median_income", "poverty_pct", "age65_pct", "renter_pct", "mobile_home_pct", "no_vehicle_pct",
               "electric_heat_pct", "median_year_built", "vacant_pct", "seasonal_pct", "dme_per_1000"],
    "flood_static": ["hand_le2m_pct", "built_hand_le2m_pct", "hand_le5m_pct", "built_hand_le5m_pct"],
    "flood_radar":  ["flood_nocrop_pct", "flood_dry_pct", "flood_built_pct", "s1_day", "s1_timely", "s1_cover_pct"],
    "embeddings":   E_COLS,
}
SETS = {
    "wind":                      ["wind"],
    "wind_rain":                 ["wind", "rain"],
    "wind_rain_place":           ["wind", "rain", "land", "people"],
    "wind_rain_place_flood":     ["wind", "rain", "land", "people", "flood_static", "flood_radar"],
    "wind_rain_place_flood_emb": ["wind", "rain", "land", "people", "flood_static", "flood_radar", "embeddings"],
    "wind_rain_emb":             ["wind", "rain", "embeddings"],
}
SET_LABELS = {"wind": "Wind", "wind_rain": "+ Rain", "wind_rain_place": "+ Place", "wind_rain_place_flood": "+ Flood",
              "wind_rain_place_flood_emb": "+ Embeddings", "wind_rain_emb": "Wind + rain\n+ embeddings"}
FULL = "wind_rain_place_flood_emb"

print(f"{len(df):,} rows (county × event), {df['fips'].nunique()} counties, {len(EVENTS)} hurricanes; outages over {LONG_H} hours in {int(df['long'].sum()):,} rows, over {VERY_LONG_H} hours in {int((df['hours_above'] >= VERY_LONG_H).sum()):,} rows; {int((df['hours_above'] == 0).sum()):,} rows never reached 10%")
group_tab = pd.DataFrame({g: {"features": len(c), "rows_with_missing": int(df[c].isna().any(axis=1).sum())} for g, c in GROUPS.items()}).T
display(group_tab)
df.groupby("event", sort=False).agg(counties=("fips", "size"), over_24h=("long", "sum"), median_hours=("hours_above", "median"),
                                    max_hours=("hours_above", "max")).round(1)

## B. Validation method and two baselines that need no model

The function `held_out` carries out the validation: for each hurricane it trains on the other 8, predicts this one, and puts the 9 sets of predictions together. With `strict=True` the training data also lose every county involved in the hurricane being predicted, so the predicted counties never appear in training at all.

The principal components of the satellite embeddings are fitted on the training data only and then applied to the hurricane being predicted.

**Metrics computed on each held-out hurricane**

| Metric | Meaning |
| --- | --- |
| `spearman` | Rank correlation between the prediction and the actual outage hours. The main metric |
| `mae_log` | Mean absolute error on log(1 + hours). It also depends on whether the model gets the overall severity of the hurricane right |
| `auc_24` | How well the classifier separates "over 24 hours" from "not over"; 0.5 = guessing, 1 = perfect |
| `auc_72` | How well the predicted duration separates "over 72 hours" |
| `ap_24` | Average precision for the over-24-hours class |

**Two baselines**

- **Rank by gust alone**: no model is trained; counties are ranked by `gust_mean_ms`. This is the minimum a model has to beat
- **History alone**: a county is predicted to lose power for as long as it did on average in the other hurricanes; counties that appear in no other hurricane get the overall mean. This measures how much "place memory" is worth on its own

In [ ]:
# 4. Leave-one-hurricane-out predictions, and the metrics
def gbt():
    return HistGradientBoostingRegressor(**GBT)

def gbt_clf():
    return HistGradientBoostingClassifier(**GBT)

def ridge():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=RIDGE_ALPHA))

def design(groups, train, test, emb_pcs=None):
    # feature matrices for training and testing. The principal components of the embeddings are fitted on the training counties only
    k = EMB_PCS if emb_pcs is None else emb_pcs
    cols = [c for g in groups for c in GROUPS[g] if g != "embeddings" or not k]
    Xtr, Xte = df.loc[train, cols], df.loc[test, cols]
    if "embeddings" in groups and k:
        uniq = df.loc[train, ["fips", "emb_year"] + E_COLS].drop_duplicates(["fips", "emb_year"])
        pca = PCA(n_components=k, random_state=SEED).fit(uniq[E_COLS].values)
        names = [f"pc{j + 1:02d}" for j in range(k)]
        Xtr = pd.concat([Xtr, pd.DataFrame(pca.transform(df.loc[train, E_COLS].values), index=train, columns=names)], axis=1)
        Xte = pd.concat([Xte, pd.DataFrame(pca.transform(df.loc[test, E_COLS].values), index=test, columns=names)], axis=1)
    return Xtr, Xte

def folds(rows, target, strict=False):
    # yields (event, training rows, test rows) hurricane by hurricane
    for ev in EVENTS:
        test = rows[(df.loc[rows, "event"] == ev).values]
        if len(test) == 0:
            continue
        train = rows[(df.loc[rows, "event"] != ev).values]
        if strict:                                                   # drop the counties being predicted
            train = train[~df.loc[train, "fips"].isin(set(df.loc[test, "fips"])).values]
        yield ev, train[df.loc[train, target].notna().values], test

def held_out(set_name, make_model=gbt, target="y", rows=None, strict=False, proba=False, emb_pcs=None):
    rows = df.index if rows is None else rows
    pred = pd.Series(np.nan, index=rows)
    for ev, train, test in folds(rows, target, strict):
        Xtr, Xte = design(SETS[set_name], train, test, emb_pcs)
        model = make_model().fit(Xtr, df.loc[train, target])
        pred.loc[test] = model.predict_proba(Xte)[:, 1] if proba else model.predict(Xte)
    return pred

def rank_corr(a, b):
    # Spearman rank correlation
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b))
    if ok.sum() < 5 or np.ptp(a[ok]) == 0 or np.ptp(b[ok]) == 0:
        return np.nan
    return float(np.corrcoef(rankdata(a[ok]), rankdata(b[ok]))[0, 1])

def auc(score, label):
    return float(roc_auc_score(label, score)) if 0 < label.sum() < len(label) else np.nan

def avg_precision(score, label):
    return float(average_precision_score(label, score)) if 0 < label.sum() < len(label) else np.nan

def event_table(score, prob=None, hours="hours_above", log_scale=True):
    # metrics on each held-out hurricane. score is the predicted log(1 + hours), or any number used for ranking; prob is the classifier's probability
    rows, out = score.dropna().index, {}
    rows = rows[df.loc[rows, hours].notna().values]
    for ev in EVENTS:
        i = rows[(df.loc[rows, "event"] == ev).values]
        if len(i) < 10:
            continue
        h, s = df.loc[i, hours].values, score.loc[i].values
        p = s if prob is None else prob.loc[i].values
        out[ev] = {"n": len(i), "spearman": rank_corr(s, h), "mae_log": float(np.mean(np.abs(s - np.log1p(h)))) if log_scale else np.nan,
                   "auc_24": auc(p, h >= LONG_H), "auc_72": auc(s, h >= VERY_LONG_H), "ap_24": avg_precision(p, h >= LONG_H)}
    return pd.DataFrame.from_dict(out, orient="index")

def summary_row(score, prob=None, hours="hours_above", log_scale=True):
    # the mean over the 9 hurricanes, and the metrics computed on all 9 pooled
    tab = event_table(score, prob, hours, log_scale)
    rows = score.dropna().index
    rows = rows[df.loc[rows, hours].notna().values]
    h, s = df.loc[rows, hours].values, score.loc[rows].values
    p, yl = (s if prob is None else prob.loc[rows].values), np.log1p(h)
    return {"rows": len(rows), "mean_spearman": tab["spearman"].mean(), "min_spearman": tab["spearman"].min(), "pooled_spearman": rank_corr(s, h),
            "pooled_mae_log": float(np.mean(np.abs(s - yl))) if log_scale else np.nan,
            "pooled_r2_log": float(1 - ((s - yl) ** 2).sum() / ((yl - yl.mean()) ** 2).sum()) if log_scale else np.nan,
            "mean_auc_24": tab["auc_24"].mean(), "pooled_auc_24": auc(p, h >= LONG_H), "mean_auc_72": tab["auc_72"].mean(),
            "pooled_ap_24": avg_precision(p, h >= LONG_H)}

# baseline 1: rank by gust alone
base_gust = df["gust_mean_ms"].astype(float)
# baseline 2: the county's mean in the other hurricanes only
base_hist = pd.Series(np.nan, index=df.index)
for ev, train, test in folds(df.index, "y"):
    county_mean = df.loc[train].groupby("fips")["y"].mean().round(9)    # rounded, so two means that are equal in exact arithmetic are not ordered by floating-point error
    base_hist.loc[test] = df.loc[test, "fips"].map(county_mean).fillna(df.loc[train, "y"].mean()).values
seen_before = pd.Series({ev: df.loc[test, "fips"].isin(set(df.loc[train, "fips"])).mean() * 100 for ev, train, test in folds(df.index, "y")})

baselines = pd.DataFrame({"gust_only": summary_row(base_gust, log_scale=False), "county_history": summary_row(base_hist)}).T
BASE_GUST = baselines.loc["gust_only", "mean_spearman"]
print(f"Share of predicted counties that also appear in the training hurricanes (%): {seen_before.round(0).to_dict()}")
tidy(baselines)

## C. Add the feature groups one by one; leave-one-hurricane-out validation

Three kinds of model are trained for each of the 6 feature sets, each with one hurricane left out at a time:

- Gradient-boosted trees predicting log(1 + hours): gives `spearman`, `mae_log`, `auc_72`
- A gradient-boosted classifier predicting whether the outage exceeds 24 hours: gives `auc_24`, `ap_24`
- Ridge regression predicting log(1 + hours): the linear baseline

In Figure 1 each gray dot is one held-out hurricane and the blue dot is the mean of the 9. The horizontal line is the "rank by gust alone" baseline.

In [ ]:
# 5. 6 feature sets × 3 kinds of model, leaving one hurricane out
pred  = {s: held_out(s, gbt) for s in SETS}                          # gradient-boosted trees, log(1 + hours)
prob  = {s: held_out(s, gbt_clf, target="long", proba=True) for s in SETS}   # probability of exceeding 24 hours
lin   = {s: held_out(s, ridge) for s in SETS}                        # ridge regression

by_event = {s: event_table(pred[s], prob[s]) for s in SETS}
summary  = pd.DataFrame({s: summary_row(pred[s], prob[s]) for s in SETS}).T
summary_ridge = pd.DataFrame({s: summary_row(lin[s]) for s in SETS}).T
summary.insert(0, "features", [sum(len(GROUPS[g]) if g != "embeddings" or not EMB_PCS else EMB_PCS for g in SETS[s]) for s in SETS])
summary["ridge_mean_spearman"] = summary_ridge["mean_spearman"]
summary.index.name = "feature_set"
summary.to_csv(f"{OUT}/step4_metrics_summary.csv")

spearman_by_event = pd.DataFrame({s: by_event[s]["spearman"] for s in SETS})
spearman_by_event.insert(0, "gust_only", event_table(base_gust, log_scale=False)["spearman"])
spearman_by_event.insert(1, "county_history", event_table(base_hist)["spearman"])
spearman_by_event.index.name = "held_out_event"
spearman_by_event.to_csv(f"{OUT}/step4_spearman_by_event.csv")
pd.concat({s: by_event[s] for s in SETS}, names=["feature_set", "held_out_event"]).to_csv(f"{OUT}/step4_metrics_by_event.csv")

out = df[["event", "fips", "name", "state", "hours_above", "affected"]].copy()
for s in SETS:
    out[f"pred_{s}"], out[f"prob24_{s}"], out[f"ridge_{s}"] = pred[s], prob[s], lin[s]
out.to_csv(f"{OUT}/step4_predictions.csv", index=False)

display(tidy(summary))
tidy(spearman_by_event)

In [ ]:
# 6. Figure 1: skill of each feature set on a hurricane it has not seen
order = list(SETS)
fig, ax = plt.subplots(figsize=(8.8, 4.5))
ax.axhline(BASE_GUST, color=FAINT, lw=1)
ax.text(1.01, BASE_GUST, f"Ranking by\ngust alone\n{BASE_GUST:.2f}", transform=ax.get_yaxis_transform(), ha="left", va="center", color=MUTED, fontsize=9)
for xi, s in enumerate(order):
    v = by_event[s]["spearman"].values
    ax.plot(xi + np.linspace(-0.13, 0.13, len(v)), v, ls="none", marker="o", ms=5, mfc=FAINT, mec=SURFACE, mew=0.8)
    ax.plot(xi, v.mean(), marker="o", ms=11, mfc=C1, mec=SURFACE, mew=2)
    ax.text(xi + 0.24, v.mean(), f"{v.mean():.2f}", va="center", color=INK, fontsize=10)
ax.axvline(len(order) - 1.5, color=GRID, lw=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels([SET_LABELS[s] for s in order]); ax.set_xlim(-0.5, len(order) - 0.45)
lo = min(float(spearman_by_event[order].min().min()), BASE_GUST) - 0.08
ax.set_ylim(max(lo, -0.2), 1.0); ax.grid(axis="x", visible=False)
ax.set_ylabel("Rank correlation, predicted vs actual duration")
ax.set_title("Skill on a hurricane the model has not seen, by feature set", pad=22)
ax.text(0, 1.03, f"Gray: each of the {len(EVENTS)} held-out hurricanes; blue: their mean. The last set swaps the place variables for embeddings",
        transform=ax.transAxes, color=MUTED, fontsize=9)
fig.savefig(f"{OUT}/step4_fig1_skill_by_feature_set.png"); plt.show()

## D. Does each group bring an improvement

Five questions, with the decision rule fixed beforehand in cell 2:

| | Question | Comparison |
| --- | --- | --- |
| H1 | Does rain contribute beyond wind | wind → wind + rain |
| H2 | Do the place variables contribute beyond wind and rain | wind + rain → plus place |
| H3 | Do the flood features contribute beyond those | plus place → plus flood |
| H4 | Do the satellite embeddings contribute beyond all hand-built features | plus flood → plus embeddings |
| H5 | Can the satellite embeddings replace the place variables | wind + rain + place against wind + rain + embeddings |

**Rule**: for H1–H4, an "improvement" needs the mean Spearman to rise by at least 0.02 and at least 6 of the 9 hurricanes to improve. For H5, the embeddings "can replace" the place variables if the set with embeddings is no more than 0.02 below the set with place variables.

Two further quantities are given for reference and play no part in the verdict:

- `p_sign_flip`: if the group were useless, the signs of the 9 differences would be random. All 512 sign combinations are tried, and this is the share whose mean difference is at least as large as the actual one
- `ci_low`, `ci_high`: a 95% interval for the mean difference from resampling counties within each hurricane. It reflects only the variation from "a different set of counties", not from "a different set of hurricanes"

In [ ]:
# 7. The five comparisons
COMPARE = [("H1", "wind", "wind_rain", "rain"), ("H2", "wind_rain", "wind_rain_place", "place"),
           ("H3", "wind_rain_place", "wind_rain_place_flood", "flood"),
           ("H4", "wind_rain_place_flood", "wind_rain_place_flood_emb", "embeddings on top"),
           ("H5", "wind_rain_place", "wind_rain_emb", "embeddings instead of place")]
rng = np.random.default_rng(SEED)
boot_rows = [[rng.choice(df.index[(df["event"] == ev).values], size=int((df["event"] == ev).sum())) for ev in EVENTS] for _ in range(N_BOOT)]

def mean_spearman_on(score, samples):
    # mean Spearman over the 9 hurricanes on the given resample
    return float(np.nanmean([rank_corr(score.loc[i].values, df.loc[i, "hours_above"].values) for i in samples]))

comp = {}
for tag, a, b, what in COMPARE:
    d = (by_event[b]["spearman"] - by_event[a]["spearman"]).reindex(EVENTS)
    flips = np.array([np.mean(d.values * np.array(sign)) for sign in itertools.product([1, -1], repeat=len(d))])
    boot = np.array([mean_spearman_on(pred[b], smp) - mean_spearman_on(pred[a], smp) for smp in boot_rows])
    gain, better = float(d.mean()), int((d > 0).sum())
    if tag == "H5":
        verdict = "can replace" if gain >= -MIN_GAIN else "cannot replace"
    else:
        verdict = "improvement" if (gain >= MIN_GAIN and better >= MIN_EVENTS) else "no improvement"
    comp[tag] = {"question": what, "from": a, "to": b, "spearman_from": by_event[a]["spearman"].mean(), "spearman_to": by_event[b]["spearman"].mean(),
                 "gain": gain, "events_better": better, "events": len(d), "p_sign_flip": float((np.abs(flips) >= abs(gain) - 1e-12).mean()),
                 "ci_low": float(np.percentile(boot, 2.5)), "ci_high": float(np.percentile(boot, 97.5)),
                 "gain_auc_24": float((by_event[b]["auc_24"] - by_event[a]["auc_24"]).mean()),
                 "gain_ridge": float(event_table(lin[b])["spearman"].mean() - event_table(lin[a])["spearman"].mean()), "verdict": verdict}
comparison = pd.DataFrame.from_dict(comp, orient="index")
comparison.index.name = "hypothesis"
comparison.to_csv(f"{OUT}/step4_feature_set_comparison.csv")
gain_by_event = pd.DataFrame({tag: (by_event[b]["spearman"] - by_event[a]["spearman"]).reindex(EVENTS) for tag, a, b, _ in COMPARE})
display(tidy(gain_by_event))
tidy(comparison, 3)

## E. Predicted against actual

This uses the reference model (wind + rain + place). Each panel is one held-out hurricane and each dot is a county. Both axes are logarithmic; the closer the dots are to the diagonal, the better the prediction. Dots that sit on one side of the diagonal as a whole mean the model overestimated or underestimated the overall severity of that hurricane, which is the hardest part for a model that has seen only 8 hurricanes.

In [ ]:
# 8. Figure 2: the reference model on each held-out hurricane, predicted against actual
TICK_H = [0, 6, 24, 72, 168, 720]
TICK_L = ["0", "6 h", "1 d", "3 d", "1 wk", "1 mo"]
ncol = 3
nrow = int(np.ceil(len(EVENTS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(11.5, 3.6 * nrow), squeeze=False, sharex=True, sharey=True)
top = np.log1p(900)
for ax, ev in zip(axes.flat, EVENTS):
    g = df[df["event"] == ev]
    ax.plot([0, top], [0, top], color=FAINT, lw=1)
    ax.scatter(pred[REFERENCE].loc[g.index].clip(0, top), g["y"], s=14, color=C1, edgecolor=SURFACE, linewidth=0.5, alpha=0.75)
    r = by_event[REFERENCE].loc[ev]
    ax.set_title(EVENT_NAMES[ev])
    ax.text(1, 1.03, f"n = {int(r['n'])}, Spearman {r['spearman']:.2f}", transform=ax.transAxes, ha="right", color=MUTED, fontsize=8.5)
    ax.set_xticks(np.log1p(TICK_H)); ax.set_xticklabels(TICK_L); ax.set_yticks(np.log1p(TICK_H)); ax.set_yticklabels(TICK_L)
    ax.set_xlim(-0.2, top); ax.set_ylim(-0.2, top)
for ax in axes.flat[len(EVENTS):]:
    ax.set_visible(False)
for ax in axes[:, 0]:
    ax.set_ylabel("Actual duration, 10%+ out")
fig.suptitle(f"Predicted against actual outage duration, reference model ({REFERENCE.replace('_', ' + ')})", x=0.01, ha="left",
             fontweight="bold", fontsize=12)
fig.supxlabel("Predicted duration (model never saw this hurricane)", color=MUTED, fontsize=10)
fig.tight_layout()
fig.savefig(f"{OUT}/step4_fig2_predicted_vs_actual.png"); plt.show()
tidy(by_event[REFERENCE])

## F. Which features matter most; gusts and forest

**Figure 3: grouped permutation importance**. This uses the model with all features. On each held-out hurricane the values of one feature group are shuffled across counties as a block (the columns of a group are shuffled together, so the combinations within the group are kept), and the drop in Spearman is recorded. The larger the drop, the more the model relies on that group. It measures "what the model uses": when two groups can express the same thing, shuffling one lets the other step in, and both look unimportant. So it has to be read together with the group-by-group results of part D.

**Figure 4: gusts, forest and outage duration**. Counties are split into thirds by forest share, and the lowest and highest thirds are kept; within gust bins, the median actual duration (dots) and the median held-out prediction of the reference model (lines) are shown. If counties with much forest lose power longer under the same gusts, orange lies above blue.

In [ ]:
# 9. Figure 3: grouped permutation importance (model with all features, computed on the held-out hurricanes)
IMP_GROUPS = {"Wind": ["wind"], "Rain": ["rain"], "Land cover and terrain": ["land"], "People and housing": ["people"],
              "Low-lying land": ["flood_static"], "Radar flood extent": ["flood_radar"], "Satellite embeddings": ["embeddings"]}
rng = np.random.default_rng(SEED)
drops = {name: [] for name in IMP_GROUPS}
for ev, train, test in folds(df.index, "y"):
    Xtr, Xte = design(SETS[FULL], train, test)
    model = gbt().fit(Xtr, df.loc[train, "y"])
    hours = df.loc[test, "hours_above"].values
    base = rank_corr(model.predict(Xte), hours)
    for name, groups in IMP_GROUPS.items():
        cols = [c for c in Xte.columns if any(c in GROUPS[g] for g in groups) or ("embeddings" in groups and c.startswith("pc"))]
        loss = []
        for _ in range(N_PERM):
            Xp = Xte.copy()
            Xp[cols] = Xte[cols].values[rng.permutation(len(Xte))]       # shuffle the whole group together
            loss.append(base - rank_corr(model.predict(Xp), hours))
        drops[name].append(float(np.mean(loss)))
importance = pd.DataFrame(drops, index=EVENTS).T
importance.insert(0, "mean_drop", importance.mean(axis=1))
importance.insert(1, "events_positive", (importance[EVENTS] > 0).sum(axis=1))
importance = importance.sort_values("mean_drop", ascending=False)
importance.index.name = "feature_group"
importance.to_csv(f"{OUT}/step4_importance.csv")

fig, ax = plt.subplots(figsize=(7.6, 0.5 * len(importance) + 1.4))
ypos = np.arange(len(importance))[::-1]
ax.barh(ypos, importance["mean_drop"], height=0.5, color=C1)
for y, v in zip(ypos, importance["mean_drop"]):
    ax.text(max(v, 0) + 0.005, y, f"{v:.3f}", va="center", color=INK, fontsize=9.5)
ax.axvline(0, color=FAINT, lw=1)
ax.set_yticks(ypos); ax.set_yticklabels(importance.index); ax.grid(axis="y", visible=False)
ax.set_xlim(min(0, float(importance["mean_drop"].min())) - 0.01, float(importance["mean_drop"].max()) * 1.18 + 0.02)
ax.set_xlabel("Drop in rank correlation when the group is shuffled")
ax.set_title("What the full model relies on", pad=22)
ax.text(0, 1.03, f"Mean over the {len(EVENTS)} held-out hurricanes, {N_PERM} shuffles each", transform=ax.transAxes, color=MUTED, fontsize=9)
fig.savefig(f"{OUT}/step4_fig3_importance.png"); plt.show()
tidy(importance, 3)

In [ ]:
# 10. Figure 4: under the same gusts, do counties with much forest lose power longer
lo_cut, hi_cut = df["forest_pct"].quantile([1 / 3, 2 / 3])
GUST_BINS = [10, 17.5, 20, 22.5, 25, 27.5, 30, 35, 50]
parts = []
for label, mask in [("Little forest", df["forest_pct"] <= lo_cut), ("Much forest", df["forest_pct"] >= hi_cut)]:
    g = df[mask]
    b = pd.cut(g["gust_mean_ms"], GUST_BINS)
    t = pd.DataFrame({"gust": g["gust_mean_ms"].groupby(b, observed=True).median(), "n": g["y"].groupby(b, observed=True).size(),
                      "actual_hours": g["hours_above"].groupby(b, observed=True).median(),
                      "predicted_hours": np.expm1(pred[REFERENCE].loc[g.index].groupby(b, observed=True).median())})
    parts.append(t[t["n"] >= 15].assign(group=label))
gust_forest = pd.concat(parts).reset_index(names="gust_bin")
gust_forest.to_csv(f"{OUT}/step4_gust_forest.csv", index=False)

fig, ax = plt.subplots(figsize=(7.4, 4.4))
for label, color in [("Little forest", C1), ("Much forest", C2)]:
    t = gust_forest[gust_forest["group"] == label]
    ax.plot(t["gust"], t["predicted_hours"] / 24, color=color, lw=2, label=f"{label}: predicted")
    ax.plot(t["gust"], t["actual_hours"] / 24, ls="none", marker="o", ms=8, mfc=color, mec=SURFACE, mew=2, label=f"{label}: actual")
    if len(t):
        ax.annotate(label, (t["gust"].iloc[-1], t["predicted_hours"].iloc[-1] / 24), xytext=(8, 0), textcoords="offset points", va="center",
                    color=INK, fontsize=9.5)
ax.set_xlim(right=ax.get_xlim()[1] + 4); ax.set_ylim(bottom=0)
ax.set_xlabel("Mean of maximum gust in the county (m/s)"); ax.set_ylabel("Median days with 10%+ out")
ax.legend(loc="upper left", fontsize=9)
ax.set_title("Outage duration by gust, for counties with little and much forest", pad=22)
ax.text(0, 1.03, f"Lowest third (forest ≤ {lo_cut:.0f}% of land) and highest third (≥ {hi_cut:.0f}%); lines are held-out predictions; bins with 15+ county-events",
        transform=ax.transAxes, color=MUTED, fontsize=8.5)
fig.savefig(f"{OUT}/step4_fig4_gust_forest.png"); plt.show()
tidy(gust_forest, 1)

## G. Robustness

Do the conclusions change with the following choices:

| Row | What is changed |
| --- | --- |
| `main` | The main result, for comparison |
| `second_denominator` | The outcome is the duration computed with the second set of customer totals (part I of Step 3a) |
| `affected_only` | Only affected counties (peak of at least 10%); same outcome |
| `affected_relative` | Only affected counties; the outcome is the relative duration `hours_rel` (which does not depend on customer totals) |
| `strict` | Strict evaluation: every county involved in the hurricane being predicted is removed from the training data |
| `ridge` | Ridge regression instead of trees |
| `shuffled_outcome` | A negative control: the outcome is shuffled across the counties of each hurricane before retraining. Features and outcome are then unrelated, so Spearman should be close to 0; a value clearly above 0 would mean information is leaking somewhere in the pipeline |

All rows except `ridge` use the feature set of the reference model and gradient-boosted trees.

**The strict evaluation matters most for the satellite embeddings**: a 64-dimensional embedding identifies a county almost uniquely, so a model could use it to memorize "how long this county lost power before". Therefore all 6 feature sets are also run under the strict evaluation and set beside the ordinary leave-one-hurricane-out results. A feature set that drops especially far under the strict evaluation owes its advantage mostly to place memory. Note also that Irma (September 2017) uses the embedding of the whole of 2017, since no earlier year exists, and that year includes the months after the storm; the other 8 hurricanes use the embedding of the year before the event.

The last table groups counties by size, to see whether the reference model does as well for small counties as for large ones.

In [ ]:
# 11. Robustness
aff = df.index[df["affected"].values]
strict_pred = {s: held_out(s, gbt, strict=True) for s in SETS}
rng = np.random.default_rng(SEED)
df["y_shuffled"] = df.groupby("event")["y"].transform(lambda v: rng.permutation(v.values))      # the outcome shuffled within each hurricane
df["hours_shuffled"] = np.expm1(df["y_shuffled"])
variants = {
    "main":               summary_row(pred[REFERENCE]),
    "second_denominator": summary_row(held_out(REFERENCE, gbt, target="y_alt"), hours="hours_above_alt"),
    "affected_only":      summary_row(held_out(REFERENCE, gbt, rows=aff)),
    "affected_relative":  summary_row(held_out(REFERENCE, gbt, target="y_rel", rows=aff), hours="hours_rel"),
    "strict":             summary_row(strict_pred[REFERENCE]),
    "ridge":              summary_row(lin[REFERENCE]),
    "shuffled_outcome":   summary_row(held_out(REFERENCE, gbt, target="y_shuffled"), hours="hours_shuffled"),
}
robust = pd.DataFrame(variants).T[["rows", "mean_spearman", "min_spearman", "pooled_spearman", "pooled_mae_log", "pooled_r2_log", "mean_auc_72"]]
robust.index.name = "variant"
robust.to_csv(f"{OUT}/step4_robustness.csv")

emb_raw = {s: event_table(held_out(s, gbt, emb_pcs=0))["spearman"].mean() for s in SETS if "embeddings" in SETS[s]}
train_rows = {False: [], True: []}
for flag in (False, True):
    train_rows[flag] = [len(train) for ev, train, test in folds(df.index, "y", strict=flag)]
strict_tab = pd.DataFrame({"standard": {s: by_event[s]["spearman"].mean() for s in SETS},
                           "strict": {s: event_table(strict_pred[s])["spearman"].mean() for s in SETS}})
strict_tab["change"] = strict_tab["strict"] - strict_tab["standard"]
strict_tab["standard_raw_64"] = pd.Series(emb_raw)                    # with the 64 embedding dimensions used as they are
strict_tab.index.name = "feature_set"
strict_tab.to_csv(f"{OUT}/step4_strict_evaluation.csv")
for s in SETS:
    out[f"strict_{s}"] = strict_pred[s]
out.to_csv(f"{OUT}/step4_predictions.csv", index=False)

size = pd.cut(df["customers_est"], [0, 5000, 20000, 100000, np.inf], labels=["<5k", "5k-20k", "20k-100k", ">100k"])
by_size = {}
for name in size.cat.categories:
    i = df.index[(size == name).values]
    by_size[name] = {"rows": len(i), "over_24h_pct": df.loc[i, "long"].mean() * 100,
                     "mean_spearman": event_table(pred[REFERENCE].loc[i])["spearman"].mean(),
                     "pooled_spearman": rank_corr(pred[REFERENCE].loc[i].values, df.loc[i, "hours_above"].values),
                     "pooled_auc_24": auc(prob[REFERENCE].loc[i].values, df.loc[i, "hours_above"].values >= LONG_H)}
by_size = pd.DataFrame.from_dict(by_size, orient="index")
by_size.index.name = "customers_est"
by_size.to_csv(f"{OUT}/step4_by_county_size.csv")

print(f"Training rows (median of the 9 folds): standard {int(np.median(train_rows[False])):,}, strict {int(np.median(train_rows[True])):,} (smallest {min(train_rows[True]):,})")
display(tidy(robust, 3))
display(tidy(strict_tab, 3))
tidy(by_size, 2)

## H. An independent check with night lights

The model is trained on EAGLE-I and also evaluated on EAGLE-I. Night lights are a completely independent second observation: if the model predicts a long outage for a county, the satellite should also see its lights dim more.

Counties with at least one clear night within 7 days of landfall are split into four groups by the held-out prediction of the reference model, and for each group the lowest night-light level, as a percentage of normal, is shown. The test is whether night lights fall further in the groups with longer predicted outages.

In [ ]:
# 12. Figure 5: are night lights dimmer in counties for which the model predicts longer outages
ntl = df[df["ntl_nights_7d"] >= 1].copy()
ntl["pred"] = pred[REFERENCE].loc[ntl.index]
ntl["dimming"] = 100 - ntl["ntl_min_pct_7d"]
QUARTERS = ["Shortest quarter", "Second", "Third", "Longest quarter"]
ntl["quarter"] = pd.qcut(ntl["pred"].rank(method="first"), 4, labels=QUARTERS)
ntl_tab = ntl.groupby("quarter", observed=True).agg(counties=("fips", "size"), predicted_hours=("pred", lambda s: float(np.expm1(s.median()))),
                                                     actual_hours=("hours_above", "median"), ntl_q25=("ntl_min_pct_7d", lambda s: s.quantile(0.25)),
                                                     ntl_median=("ntl_min_pct_7d", "median"), ntl_q75=("ntl_min_pct_7d", lambda s: s.quantile(0.75)))
ntl_tab.to_csv(f"{OUT}/step4_ntl_check.csv")
rho_ntl_pred = rank_corr(ntl["pred"].values, ntl["dimming"].values)
rho_ntl_obs  = rank_corr(ntl["hours_above"].values, ntl["dimming"].values)
rho_ntl_by_event = pd.Series({ev: rank_corr(g["pred"].values, g["dimming"].values) for ev, g in ntl.groupby("event", sort=False)})

fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(len(ntl_tab))
ax.axhline(100, color=FAINT, lw=1)
ax.vlines(x, ntl_tab["ntl_q25"], ntl_tab["ntl_q75"], color=C2, lw=6, alpha=0.35)
ax.plot(x, ntl_tab["ntl_median"], ls="none", marker="o", ms=9, mfc=C2, mec=SURFACE, mew=2)
for xi, (med, n) in enumerate(zip(ntl_tab["ntl_median"], ntl_tab["counties"])):
    ax.text(xi + 0.12, med, f"{med:.0f}%", va="center", color=INK, fontsize=10)
    ax.text(xi, -0.115, f"n = {int(n):,}", ha="center", va="top", color=MUTED, fontsize=9, transform=ax.get_xaxis_transform())
ax.set_xticks(x); ax.set_xticklabels(ntl_tab.index); ax.set_xlim(-0.5, len(x) - 0.4); ax.set_ylim(0, 140); ax.grid(axis="x", visible=False)
ax.set_xlabel("Counties grouped by predicted outage duration (model never saw the hurricane)", labelpad=32)
ax.set_ylabel("Lowest night lights within 7 days,\npercent of normal")
ax.set_title("Night lights, an independent observation, against the model's prediction", pad=22)
ax.text(0, 1.03, f"Median and middle half of counties; Spearman of prediction and dimming {rho_ntl_pred:.2f} (actual duration and dimming: {rho_ntl_obs:.2f})",
        transform=ax.transAxes, color=MUTED, fontsize=8.5)
fig.savefig(f"{OUT}/step4_fig5_night_lights.png"); plt.show()
print(f"Spearman correlation of predicted duration and night-light dimming: all {len(ntl):,} county-events, {rho_ntl_pred:.2f}; by event: {rho_ntl_by_event.round(2).to_dict()}")
tidy(ntl_tab, 1)

## I. Checklist

The thresholds and the decision rule were all written down before running. The first 6 items are quality checks; the last 5 are the answers to the five questions of part D.

In [ ]:
# 13. Step 4 checklist
def verdict(passed):
    return "pass" if passed else "fail"

ref  = summary.loc[REFERENCE]
wind = summary.loc["wind"]
gap_alt = abs(robust.loc["second_denominator", "mean_spearman"] - robust.loc["main", "mean_spearman"])
checks = [
    ("Quality", "Wind-only model, mean Spearman", round(wind["mean_spearman"], 2), "≥ 0.60", verdict(wind["mean_spearman"] >= 0.60)),
    ("Quality", "Mean Spearman of the reference model minus that of ranking by gust alone", round(ref["mean_spearman"] - BASE_GUST, 3), "> 0", verdict(ref["mean_spearman"] > BASE_GUST)),
    ("Quality", "AUC of the reference model for exceeding 24 hours (mean of 9)", round(ref["mean_auc_24"], 2), "≥ 0.80", verdict(ref["mean_auc_24"] >= 0.80)),
    ("Quality", "Change in mean Spearman with the second denominator (absolute)", round(float(gap_alt), 3), "≤ 0.05", verdict(gap_alt <= 0.05)),
    ("Quality", "Mean Spearman of the reference model under the strict evaluation", round(robust.loc["strict", "mean_spearman"], 2), "≥ 0.50", verdict(robust.loc["strict", "mean_spearman"] >= 0.50)),
    ("Quality", "Spearman correlation of the reference model's prediction and night-light dimming", round(rho_ntl_pred, 2), "≥ 0.20", verdict(rho_ntl_pred >= 0.20)),
]
for tag, a, b, what in COMPARE:
    r = comparison.loc[tag]
    rule = f"≥ {-MIN_GAIN}" if tag == "H5" else f"≥ {MIN_GAIN}, {MIN_EVENTS}+ / {int(r['events'])}"
    checks.append((tag, f"{what}: {a} → {b}", f"{r['gain']:+.3f}, {int(r['events_better'])} / {int(r['events'])}", rule, r["verdict"]))
check_tab = pd.DataFrame(checks, columns=["Part", "Indicator", "Value", "Threshold", "Verdict"])
check_tab.to_csv(f"{OUT}/step4_checklist.csv", index=False)
check_tab

## Output files

All written to the `us_flood_outage` folder:

- `step4_checklist.csv`, `step4_metrics_summary.csv`, `step4_spearman_by_event.csv`, `step4_metrics_by_event.csv`
- `step4_feature_set_comparison.csv`, `step4_importance.csv`, `step4_gust_forest.csv`
- `step4_robustness.csv`, `step4_strict_evaluation.csv`, `step4_by_county_size.csv`, `step4_ntl_check.csv`
- `step4_predictions.csv`
- The 5 figures: `step4_fig1_skill_by_feature_set.png` to `step4_fig5_night_lights.png`

**On versions**: the reported numbers were produced with scikit-learn 1.6.1 (cell 1 prints the versions in use). With another version the gradient-boosted-tree numbers change in the second or third decimal and the ridge numbers do not; the answers to the five questions are the same under 1.6.1 and 1.9.1.